In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pwd

/oscar/data/epavlick/zyang220/function_vectors


In [3]:
# kill -9 PID
!nvidia-smi

Sun Feb 23 21:41:56 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L40S                    On  | 00000000:61:00.0 Off |                    0 |
| N/A   34C    P0              83W / 350W |   1842MiB / 46068MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [24]:
import os, re, json
import torch, numpy as np
from src.utils.plotly_utils import imshow

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import (get_mean_head_activations, 
    compute_universal_function_vector, compute_function_vector,
)
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval
from src.compute_indirect_effect import compute_indirect_effect

# Load model 

In [ ]:
model_name = 'gpt2-medium'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
 # the layer to add the funciton vector to 

Loading:  gpt2-medium


# Load dataset and Compute task-conditioned mean activations

In [7]:
dataset = load_dataset('capitalize', seed=0)
mean_activations = get_mean_head_activations(dataset, model, model_config, tokenizer)
# Shape of the mean activations is 'n_layer n_head n_token d_head'

In [8]:
mean_activations.shape 

torch.Size([24, 16, 97, 64])

# Compute function vector (FV) from task-specific attention heads

In [9]:
n_shots  = 10 
n_trials = 25
last_token_only = True 
prefixes = {"input":"Q:", "output":"A:", "instructions":""}
separators = {"input":"\n", "output":"\n\n", "instructions":""}
indirect_effect = compute_indirect_effect(dataset, mean_activations, model=model, 
    model_config=model_config, tokenizer=tokenizer, 
    n_shots=n_shots, n_trials=n_trials, last_token_only=last_token_only, 
    prefixes=prefixes, separators=separators)
indirect_effect.shape

  0%|                                                                                                                       | 0/25 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:15<00:00,  3.00s/it]


torch.Size([25, 24, 16])

In [21]:
imshow(
    indirect_effect.mean(dim=0).T, 
    title = "Average indirect effect of function-vector intervention on uppercasing task",
    width = 1000,
    height = 600,
    labels = {"x": "Layer", "y": "Head"},
    aspect = "equal",
)

# Compute function vector (FV) from shared attention heads 

In [ ]:
# not run because it needs pre-calculated AIE for all tasks 
# FV, top_heads = compute_universal_function_vector(mean_activations, model, model_config, n_top_heads=10)

# Prompt Creation - ICL, Shuffled-Label, Zero-Shot, and Natural Text

In [27]:
# Sample ICL example pairs, and a test word
dataset = load_dataset('capitalize')
word_pairs = dataset['train'][:5]
test_pair = dataset['test'][21]

prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True)
sentence = create_prompt(prompt_data)
print("ICL prompt:\n", repr(sentence), '\n\n')

shuffled_prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
shuffled_sentence = create_prompt(shuffled_prompt_data)
print("Shuffled ICL Prompt:\n", repr(shuffled_sentence), '\n\n')

zeroshot_prompt_data = word_pairs_to_prompt_data({'input':[], 'output':[]}, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
zeroshot_sentence = create_prompt(zeroshot_prompt_data)
print("Zero-Shot Prompt:\n", repr(zeroshot_sentence))

ICL prompt:
 '<|endoftext|>Q: dance\nA: Dance\n\nQ: soda\nA: Soda\n\nQ: before\nA: Before\n\nQ: youthful\nA: Youthful\n\nQ: orange\nA: Orange\n\nQ: turkey\nA:' 


Shuffled ICL Prompt:
 '<|endoftext|>Q: dance\nA: Soda\n\nQ: soda\nA: Orange\n\nQ: before\nA: Before\n\nQ: youthful\nA: Youthful\n\nQ: orange\nA: Dance\n\nQ: turkey\nA:' 


Zero-Shot Prompt:
 '<|endoftext|>Q: turkey\nA:'


# Evaluation - 10FV

In [26]:
FV, top_heads = compute_function_vector(mean_activations, indirect_effect, model, model_config, n_top_heads=10)

In [123]:
EDIT_LAYER = 5

In [124]:
# Perform an intervention on the shuffled setting
clean_logits, interv_logits = function_vector_intervention(
    shuffled_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(shuffled_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: dance\nA: Soda\n\nQ: soda\nA: Orange\n\nQ: before\nA: Before\n\nQ: youthful\nA: Youthful\n\nQ: orange\nA: Dance\n\nQ: turkey\nA:' 

Input Query: 'turkey', Target: 'Turkey'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [(' Turkey', 0.04972), (' turkey', 0.02464), (' Before', 0.02014), (' After', 0.01117), (' Chicken', 0.01087)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [(' Turkey', 0.04245), (' Before', 0.02764), ('\n', 0.02172), (' turkey', 0.02122), (' Chicken', 0.01838)]


In [125]:
# Intervention on the zero-shot prompt
clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, 
    [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(zeroshot_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: turkey\nA:' 

Input Query: 'turkey', Target: 'Turkey'

Zero-Shot Top K Vocab Probs:
 [(' I', 0.10837), (' It', 0.03141), (' Yes', 0.02514), (' No', 0.01913), (' You', 0.01812)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [(' turkey', 0.45845), (' chicken', 0.03406), (' tur', 0.0117), ('\n', 0.01031), (' Turkey', 0.00753)]


In [126]:
sentence = f"The word \"{test_pair['input']}\" means"
co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)

print("Input Sentence: ", repr(sentence))
print("GPT2-medium:" , repr(tokenizer.decode(co.squeeze())))
print("GPT2-medium+FV:", repr(tokenizer.decode(io.squeeze())), '\n')

Input Sentence:  'The word "turkey" means'
GPT2-medium: 'The word "turkey" means "to eat" in the Old English language.'
GPT2-medium+FV: 'The word "turkey" means "turkey" in the Bible.\n\n' 



# Eval - 5 FV

In [128]:
FV, top_heads = compute_function_vector(mean_activations, indirect_effect, model, model_config, n_top_heads=5)

In [129]:
EDIT_LAYER = 5

In [130]:
# Perform an intervention on the shuffled setting
clean_logits, interv_logits = function_vector_intervention(
    shuffled_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(shuffled_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: dance\nA: Soda\n\nQ: soda\nA: Orange\n\nQ: before\nA: Before\n\nQ: youthful\nA: Youthful\n\nQ: orange\nA: Dance\n\nQ: turkey\nA:' 

Input Query: 'turkey', Target: 'Turkey'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [(' Turkey', 0.04972), (' turkey', 0.02464), (' Before', 0.02014), (' After', 0.01117), (' Chicken', 0.01087)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [(' Turkey', 0.04311), (' turkey', 0.02479), (' Before', 0.02087), (' Chicken', 0.0141), ('\n', 0.01398)]


In [131]:
# Intervention on the zero-shot prompt
clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, 
    [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(zeroshot_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: turkey\nA:' 

Input Query: 'turkey', Target: 'Turkey'

Zero-Shot Top K Vocab Probs:
 [(' I', 0.10837), (' It', 0.03141), (' Yes', 0.02514), (' No', 0.01913), (' You', 0.01812)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [(' turkey', 0.14773), (' I', 0.04438), (' It', 0.01816), (' chicken', 0.0178), (' The', 0.01289)]


In [132]:
sentence = f"The word \"{test_pair['input']}\" means"
co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)

print("Input Sentence: ", repr(sentence))
print("GPT2-medium:" , repr(tokenizer.decode(co.squeeze())))
print("GPT2-medium+FV:", repr(tokenizer.decode(io.squeeze())), '\n')

Input Sentence:  'The word "turkey" means'
GPT2-medium: 'The word "turkey" means "to eat" in the Old English language.'
GPT2-medium+FV: 'The word "turkey" means "turkey" in the United States.\n' 

